# Compressor threshold calibration - v2 study

**What this is:** constrained offline calibration of `WINDOW_MEAN_MIN` / `WINDOW_P5_MIN` / `WINDOW_MIN_MIN` with a hard false-accept constraint, honest held-out validation, and bootstrap stability. It is NOT a trained model and its output is research evidence only.

**What changed vs the v1 Optuna study** (per the 2026-07-19 adversarial review):
1. Real holdout - v1's GroupKFold averaged fixed thresholds over all folds, which equals in-sample evaluation.
2. Exhaustive deterministic grid instead of TPE sampling - reports the whole tied-optimal set, no sampler noise, no seed sensitivity.
3. `PROBE_MIN_SOURCE_BITS_PER_PIXEL` removed from the search - structurally unidentifiable from this data (no positive below bpp 0.1015); stays at 0.03.
4. Dataset hygiene - dedup to one row per source (v1 had 3.8x duplication), pre-pairing-fix rows superseded by post-fix re-measures, proven-false positives (133776.mp4) excluded, misalignment / cert-unavailable skips no longer counted as quality negatives.
5. False acceptance is a hard constraint (FA=0), not a 100x cost; movement from current production is tie-broken toward minimal change and never rewarded.
6. `NO_CHANGE_RECOMMENDED` is a first-class outcome - the study only emits a Kotlin candidate if every held-out fold agrees current production is outside the tied-optimal set.

**How to run:** Runtime > Run all. When prompted, upload `Compressor_Optuna_V2_Study.zip`. Results download automatically at the end.

In [ ]:
# 1) Dependencies (no optuna needed - the v2 study is an exhaustive grid)
%pip install -q "pandas>=2.0" "numpy>=1.26" "scikit-learn>=1.4"
import pandas, numpy, sklearn
print('pandas', pandas.__version__, '| numpy', numpy.__version__, '| sklearn', sklearn.__version__)

In [ ]:
# 2) Upload and unpack the study bundle
import os, zipfile
if not os.path.exists('Compressor_Optuna_V2_Study'):
    from google.colab import files
    uploaded = files.upload()  # upload Compressor_Optuna_V2_Study.zip
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as z:
        z.extractall('.')
root = 'Compressor_Optuna_V2_Study'
assert os.path.exists(f'{root}/prepared_data/all_jobs_normalized.csv'), 'bundle incomplete'
print('bundle ready')

In [ ]:
# 3) Prepare the v2 dataset (hygiene + dedup). Review prep_report_v2.json output below.
!python {root}/scripts/prepare_dataset_v2.py

In [ ]:
# 4) Run the v2 study (exhaustive grid + nested holdout + bootstrap; a few minutes)
!python {root}/scripts/run_study_v2.py

In [ ]:
# 5) Show the human-readable summary and download everything
print(open(f'{root}/results_v2/study_report_v2.md').read())
import shutil
shutil.make_archive('Compressor_Optuna_V2_Results', 'zip', f'{root}/results_v2')
# include the prep report so the results zip is self-explaining
import zipfile
with zipfile.ZipFile('Compressor_Optuna_V2_Results.zip', 'a') as z:
    z.write(f'{root}/prepared_data/prep_report_v2.json', 'prep_report_v2.json')
    z.write(f'{root}/prepared_data/model_rows_v2.csv', 'model_rows_v2.csv')
from google.colab import files
files.download('Compressor_Optuna_V2_Results.zip')